In [1]:
#Python Modules
import os
import pandas as pd
from sklearn.model_selection import train_test_split
import torch 
from torch.utils.data import DataLoader

#os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
#os.environ["TORCH_USE_CUDA_DSA"]   = "1"

#ML Modules
import train.train_code as tr_c
import preprocess.data_frame_preprocess as prep
import models.model_bge_m3 as bge_m3
import models.model_xgboost as xgb_m3
import evaluate
import dataset.Train as multi_task

#Config FILE
import config as cfg

/home/andreoseo/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Transformer

## Check Save Paths

In [2]:
os.makedirs(cfg.paths.SAVE_DIR,exist_ok=True)

In [3]:
raw_df = pd.read_csv(cfg.paths.CSV_PATH)
df = prep.prepare_dataframe(raw_df,cfg) #New columns: lyrics_clean & gloss_clean

strat_key = (df['label_t1'].astype(str) + '_' +
             df['label_t3'].astype(str) + '_' +
             df[cfg.columns.LABEL_COLS_T2].astype(str).agg('-'.join, axis=1))

[METRICS] Palabras promedio PRE:  348.05
[METRICS] Palabras promedio POST: 295.09
[METRICS] Reducción de tokens:    15.22%
[FLAG] Usando Glosses para entrenamiento


## Load Models

In [4]:
#Load Transformer Model
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer,model = bge_m3.load_tokenizer_model(cfg,device)

Loading weights: 100%|█████████████████████████████████████████████████████████████| 391/391 [00:00<00:00, 92606.75it/s]


## Create Datasets

In [5]:
# Generate DataSets and DataLoaders
train_df, val_df = train_test_split(df, test_size=cfg.train.TEST_SIZE,
                                    stratify=strat_key,
                                    random_state= cfg.train.SEED)

#Dataset
train_ds = multi_task.MultiTaskDataset(train_df, cfg,tokenizer, cfg.tokenization.MAX_LEN,
                            head_len=cfg.tokenization.HEAD_LEN,
                            model_prefix=cfg.paths.MODEL_PREFIX)
val_ds   = multi_task.MultiTaskDataset(val_df, cfg,  tokenizer, cfg.tokenization.MAX_LEN,
                            head_len=cfg.tokenization.HEAD_LEN,
                            model_prefix=cfg.paths.MODEL_PREFIX)
#DataLoader
train_loader = DataLoader(train_ds, batch_size=cfg.train.BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.train.BATCH_SIZE,
                          num_workers=8, pin_memory=True)

In [6]:
if cfg.tokenization.HEAD_LEN is False:
    print(f'  Truncación: solo cabeza (primeros {cfg.tokenization.MAX_LEN} tokens)')
else:
    print(f'  Truncación: Head+Tail  (head={cfg.tokenization.HEAD_LEN} | tail={cfg.tokenization.MAX_LEN - cfg.tokenization.HEAD_LEN - 2})')

  Truncación: solo cabeza (primeros 1024 tokens)


# Train XGBOOST w Embeddings

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Cargar pesos entrenados
#tokenizer, model = load_tokenizer_model(cfg, device)
model.load_state_dict(
    torch.load(
        os.path.join(
            '/home/andreoseo/Documents/MiSongGyny2026/models/checkpoints_true/bg3_v3_only_t2',
            'model_weights.pt'
        ),
        map_location=device
    )
)

<All keys matched successfully>

In [8]:
X_train, y_t1_train, y_t2_train, y_t3_train = xgb_m3.extract_embeddings_and_labels(model, train_loader, device)
X_val,   y_t1_val,   y_t2_val,   y_t3_val   = xgb_m3.extract_embeddings_and_labels(model, val_loader,   device)
#X_test_emb  = extract_embeddings(model, test_loader,  device)

In [10]:
from sklearn.multiclass import OneVsRestClassifier
import xgboost as xgb
import optuna
from sklearn.metrics import f1_score, precision_score, classification_report


xgb_base = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    eval_metric='logloss',
    device='cuda',
    random_state=42
)

xgb_t2 = OneVsRestClassifier(xgb_base)
xgb_t2.fit(X_train, y_t2_train)  # y_t2_train shape: (N, 4) con 0s y 1s

print(classification_report(y_t2_val, xgb_t2.predict(X_val)))

              precision    recall  f1-score   support

           0       0.82      0.85      0.83       241
           1       0.54      0.47      0.50        95
           2       0.63      0.70      0.66       178
           3       0.87      0.82      0.85       388

   micro avg       0.77      0.77      0.77       902
   macro avg       0.71      0.71      0.71       902
weighted avg       0.77      0.77      0.77       902
 samples avg       0.77      0.77      0.76       902



/home/andreoseo/miniconda3/lib/python3.13/site-packages/xgboost/core.py:751: UserWarning: [01:28:34] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
/home/andreoseo/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [11]:
import optuna
import xgboost as xgb
import numpy as np
from sklearn.metrics import f1_score, classification_report

optuna.logging.set_verbosity(optuna.logging.WARNING)

def train_binary_clf(X_train, y_train, X_val, y_val, n_trials=50, device='cuda'):
    """Entrena un clasificador binario XGBoost con Optuna para balanceo."""
    neg = (y_train == 0).sum()
    pos = (y_train == 1).sum()
    spw = neg / pos if pos > 0 else 1.0

    def objective(trial):
        clf = xgb.XGBClassifier(
            n_estimators     = trial.suggest_int  ('n_estimators',       100, 800),
            max_depth        = trial.suggest_int  ('max_depth',            2,   8),
            learning_rate    = trial.suggest_float('learning_rate',    0.005, 0.1, log=True),
            subsample        = trial.suggest_float('subsample',          0.5, 1.0),
            colsample_bytree = trial.suggest_float('colsample_bytree',   0.5, 1.0),
            min_child_weight = trial.suggest_int  ('min_child_weight',     1,  20),
            scale_pos_weight = spw,
            eval_metric      = 'logloss',
            device           = device,
            random_state     = 42
        )
        clf.fit(X_train, y_train)
        return f1_score(y_val, clf.predict(X_val))

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True,n_jobs=14)

    # reentrenar con mejores params
    best_clf = xgb.XGBClassifier(
        **study.best_params,
        scale_pos_weight = spw,
        eval_metric      = 'logloss',
        device           = device,
        random_state     = 42
    )
    best_clf.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    return best_clf, study.best_value, study.best_params


def train_all_t2(X_train, y_t2_train, X_val, y_t2_val, label_cols,
                 optuna_cols=None, n_trials=50, device='cuda'):
    """
    Entrena un clasificador por cada columna de t2.
    optuna_cols: lista de columnas donde aplicar Optuna. 
                 Si None, aplica Optuna a todas.
                 Ejemplo: ['type_violence']
    """
    if optuna_cols is None:
        optuna_cols = label_cols

    clfs_t2 = {}

    for i, col in enumerate(label_cols):
        y_tr = y_t2_train[:, i]
        y_vl = y_t2_val[:,   i]
        neg  = (y_tr == 0).sum()
        pos  = (y_tr == 1).sum()
        spw  = neg / pos if pos > 0 else 1.0

        print(f"\n── {col} | pos={pos} neg={neg} spw={spw:.2f} ──")

        if col in optuna_cols:
            print(f"  → Optuna ({n_trials} trials)")
            clf, best_f1, best_params = train_binary_clf(
                X_train, y_tr, X_val, y_vl, n_trials=n_trials, device=device
            )
            print(f"  → Mejor f1={best_f1:.4f} | params={best_params}")
        else:
            print(f"  → XGBoost estándar")
            clf = xgb.XGBClassifier(
                n_estimators     = 300,
                max_depth        = 6,
                learning_rate    = 0.05,
                eval_metric      = 'logloss',
                scale_pos_weight = spw,
                device           = device,
                random_state     = 42
            )
            clf.fit(X_train, y_tr, eval_set=[(X_val, y_vl)], verbose=False)

        clfs_t2[col] = clf

    # evaluación final
    pred_t2 = np.column_stack([clfs_t2[col].predict(X_val) for col in label_cols])
    print("\n── Classification Report T2 ──")
    print(classification_report(y_t2_val, pred_t2, target_names=label_cols))

    return clfs_t2


    

In [12]:
# solo violence con Optuna, el resto estándar
"""clfs_t2 = train_all_t2(
    X_train, y_t2_train,
    X_val,   y_t2_val,
    label_cols   = cfg.columns.LABEL_COLS_T2,
    optuna_cols  = ['type_violence'],
    n_trials     = 150,
    device       = 'cuda'
)"""

# o todas con Optuna
clfs_t2 = train_all_t2(
    X_train, y_t2_train,
    X_val,   y_t2_val,
    label_cols  = cfg.columns.LABEL_COLS_T2,
    n_trials    = 200,
    device      = 'cuda'
)


── type_sexualization | pos=563 neg=1049 spw=1.86 ──
  → Optuna (200 trials)


Best t


  → Mejor f1=0.8364 | params={'n_estimators': 689, 'max_depth': 3, 'learning_rate': 0.08496076688110663, 'subsample': 0.9010381138257196, 'colsample_bytree': 0.8839283656166387, 'min_child_weight': 7}

── type_violence | pos=219 neg=1393 spw=6.36 ──
  → Optuna (200 trials)


Best t


  → Mejor f1=0.5616 | params={'n_estimators': 765, 'max_depth': 3, 'learning_rate': 0.031480261016398114, 'subsample': 0.700520784629488, 'colsample_bytree': 0.6322597127923733, 'min_child_weight': 17}

── type_hate | pos=411 neg=1201 spw=2.92 ──
  → Optuna (200 trials)


Best t


  → Mejor f1=0.6970 | params={'n_estimators': 538, 'max_depth': 7, 'learning_rate': 0.07472446384807012, 'subsample': 0.9698935194075452, 'colsample_bytree': 0.8862967486828934, 'min_child_weight': 19}

── nothing | pos=906 neg=706 spw=0.78 ──
  → Optuna (200 trials)


Best t


  → Mejor f1=0.8522 | params={'n_estimators': 439, 'max_depth': 7, 'learning_rate': 0.07136981004554752, 'subsample': 0.538189510160689, 'colsample_bytree': 0.7618655986828962, 'min_child_weight': 6}

── Classification Report T2 ──
                    precision    recall  f1-score   support

type_sexualization       0.81      0.86      0.84       241
     type_violence       0.53      0.60      0.56        95
         type_hate       0.63      0.78      0.70       178
           nothing       0.87      0.83      0.85       388

         micro avg       0.76      0.80      0.78       902
         macro avg       0.71      0.77      0.74       902
      weighted avg       0.77      0.80      0.79       902
       samples avg       0.77      0.80      0.77       902



/home/andreoseo/miniconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [16]:
clfs_t2

{'type_sexualization': XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=0.8839283656166387, device='cuda',
               early_stopping_rounds=None, enable_categorical=False,
               eval_metric='logloss', feature_types=None, feature_weights=None,
               gamma=None, grow_policy=None, importance_type=None,
               interaction_constraints=None, learning_rate=0.08496076688110663,
               max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
               max_delta_step=None, max_depth=3, max_leaves=None,
               min_child_weight=7, missing=nan, monotone_constraints=None,
               multi_strategy=None, n_estimators=689, n_jobs=None,
               num_parallel_tree=None, ...),
 'type_violence': XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsa

In [15]:
import joblib

In [17]:
output_dir_aux = '/home/andreoseo/Documents/MiSongGyny2026/models/checkpoints_true/xgboost_bg3_v3_t1_t3/'
# guardar todos los clasificadores de t2 (incluido el violence optimizado)
for col, clf in clfs_t2.items():
    joblib.dump(clf, f'{output_dir_aux}/xgb_t2_{col}.pkl')


In [ ]:
x_